# ✈️ Phase 4 — Machine Learning Model, Results & Discussion

**Course:** COMP4381 Data Science and Analytics  
**Instructor:** Ahmed Sabbah  
**Dataset:** Cleaned flight punctuality data for 6 European countries (FR, DE, IT, ES, NL, GB)  
**Target Variable:** `Is_Delayed` — binary classification (1 = delayed, 0 = on time)

---

This notebook covers:
1. Loading the cleaned dataset from Phase 3
2. Feature selection and encoding
3. Training a Decision Tree classifier
4. Evaluating the model with accuracy, precision, recall, F1, and confusion matrix
5. Feature importance analysis
6. Discussion, conclusion, and limitations


## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

print("Libraries loaded successfully.")


## 2. Load the Cleaned Dataset

We load `cleaned_flight_data.csv` from the processed data folder — the output of Phase 3.  
This dataset has already been cleaned, merged with airport metadata, and feature-engineered.


In [ ]:
df = pd.read_csv('../data/processed/cleaned_flight_data.csv')
print(f"Dataset shape: {df.shape}")
print(f"\nColumn names:")
print(list(df.columns))


In [ ]:
df.head(5)


## 3. Feature Selection

We select the features that will be used to train the model. We exclude:
- `Date` and `Airport` — identifiers, not predictive features
- `municipality` — same information already represented by `Airport`
- `DayOfWeek` — encoded as `Weekend`, which captures the key distinction
- `AverageDelay` — this is the raw value from which `Is_Delayed` was derived; including it would cause **data leakage**
- Punctuality percentage columns stored as strings (require additional parsing)

**Features used:**
| Feature | Type | Reason |
|---|---|---|
| `Avg Departure Schedule Delay` | Numeric | Key delay signal |
| `Avg Arrival Schedule Delay` | Numeric | Key delay signal |
| `Avg Departure - Arrival Schedule Delay` | Numeric | Difference pattern |
| `latitude_deg` | Numeric | Geography |
| `longitude_deg` | Numeric | Geography |
| `elevation_ft` | Numeric | Airport characteristic |
| `Month` | Numeric | Seasonal pattern |
| `Weekend` | Binary | Traffic pattern |
| `iso_country` | Categorical (encoded) | Country effect |
| `Season` | Categorical (encoded) | Seasonal pattern |
| `Airport_Size` | Categorical (encoded) | Airport capacity |


In [ ]:
# Numeric features
numeric_features = [
    'Avg Departure Schedule Delay',
    'Avg Arrival Schedule Delay',
    'Avg Departure - Arrival Schedule Delay',
    'latitude_deg',
    'longitude_deg',
    'elevation_ft',
    'Month',
    'Weekend'
]

# Categorical features to one-hot encode
categorical_features = ['iso_country', 'Season', 'Airport_Size']

# Target
target = 'Is_Delayed'

# Build feature matrix
X_numeric = df[numeric_features].copy()
X_categorical = pd.get_dummies(df[categorical_features], drop_first=False)
X = pd.concat([X_numeric, X_categorical], axis=1)
y = df[target]

print(f"Feature matrix shape: {X.shape}")
print(f"Target distribution:")
print(y.value_counts())
print(f"\nClass balance:")
print(y.value_counts(normalize=True).round(3))


## 4. Train / Test Split

We split the dataset into **80% training** and **20% testing** using a fixed random seed for reproducibility.  
Stratification ensures that the class ratio (delayed vs. on-time) is preserved in both splits.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f"Training set size : {X_train.shape[0]:,} rows")
print(f"Testing set size  : {X_test.shape[0]:,} rows")


## 5. Train the Decision Tree Classifier

We use a **Decision Tree** — a supervised learning algorithm that splits the data on feature thresholds to minimize impurity.  
It was taught in this course and is interpretable, making it easy to explain the model's decisions.

**Hyperparameters chosen:**
- `max_depth=5` — limits the tree to 5 levels, which prevents overfitting on the training set
- `criterion='gini'` — uses Gini impurity, the standard for classification trees
- `random_state=42` — ensures reproducible results


In [ ]:
model = DecisionTreeClassifier(max_depth=5, criterion='gini', random_state=42)
model.fit(X_train, y_train)
print("Model trained successfully.")


## 6. Make Predictions

In [ ]:
y_pred = model.predict(X_test)
print("Predictions generated.")


## 7. Model Evaluation

We evaluate the model using four standard metrics:

| Metric | Formula | What it measures |
|---|---|---|
| **Accuracy** | Correct predictions / Total predictions | Overall correctness |
| **Precision** | TP / (TP + FP) | Among predicted delays, how many were real? |
| **Recall** | TP / (TP + FN) | Among real delays, how many did we catch? |
| **F1 Score** | 2 × (Precision × Recall) / (Precision + Recall) | Harmonic mean of precision and recall |


In [ ]:
accuracy  = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall    = recall_score(y_test, y_pred)
f1        = f1_score(y_test, y_pred)

print("=" * 40)
print("       MODEL EVALUATION RESULTS")
print("=" * 40)
print(f"  Accuracy  : {accuracy:.4f}  ({accuracy*100:.2f}%)")
print(f"  Precision : {precision:.4f}")
print(f"  Recall    : {recall:.4f}")
print(f"  F1 Score  : {f1:.4f}")
print("=" * 40)


In [ ]:
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['On Time (0)', 'Delayed (1)']))


## 8. Confusion Matrix

The confusion matrix shows how the model's predictions break down into four categories:
- **True Negatives (TN):** correctly predicted on-time
- **False Positives (FP):** predicted delayed but actually on-time
- **False Negatives (FN):** predicted on-time but actually delayed
- **True Positives (TP):** correctly predicted delayed


In [ ]:
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, interpolation='nearest', cmap='Blues')
plt.colorbar(im, ax=ax)

tick_marks = [0, 1]
ax.set_xticks(tick_marks)
ax.set_yticks(tick_marks)
ax.set_xticklabels(['On Time (0)', 'Delayed (1)'], fontsize=11)
ax.set_yticklabels(['On Time (0)', 'Delayed (1)'], fontsize=11)

thresh = cm.max() / 2
for i in range(2):
    for j in range(2):
        ax.text(j, i, format(cm[i, j], 'd'),
                ha='center', va='center',
                color='white' if cm[i, j] > thresh else 'black',
                fontsize=14, fontweight='bold')

ax.set_ylabel('Actual Label', fontsize=12)
ax.set_xlabel('Predicted Label', fontsize=12)
ax.set_title('Confusion Matrix — Decision Tree (max_depth=5)', fontsize=13, pad=12)
plt.tight_layout()
plt.savefig('../figures/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nTrue Negatives  (TN): {tn:,}")
print(f"False Positives (FP): {fp:,}")
print(f"False Negatives (FN): {fn:,}")
print(f"True Positives  (TP): {tp:,}")


## 9. Feature Importance

Decision trees compute an importance score for each feature based on how much it reduces impurity across all splits.  
A higher score means the feature contributes more to the model's decisions.


In [ ]:
importances = model.feature_importances_
feature_names = X.columns.tolist()

importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values('Importance', ascending=False).reset_index(drop=True)

print("Top 15 most important features:")
print(importance_df.head(15).to_string(index=False))


In [ ]:
top_n = 15
top_features = importance_df.head(top_n)

fig, ax = plt.subplots(figsize=(9, 6))
colors = ['#1f77b4' if i < 3 else '#aec7e8' for i in range(top_n)]
bars = ax.barh(top_features['Feature'][::-1], top_features['Importance'][::-1],
               color=colors[::-1], edgecolor='white')

ax.set_xlabel('Importance Score', fontsize=12)
ax.set_title(f'Top {top_n} Feature Importances — Decision Tree', fontsize=13)
ax.grid(axis='x', alpha=0.4)
for bar, val in zip(bars, top_features['Importance'][::-1]):
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('../figures/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()


## 10. Effect of Tree Depth on Accuracy

To understand whether `max_depth=5` is a good choice, we train the same Decision Tree at different depths and compare training vs. testing accuracy.  
This reveals the **bias-variance trade-off**: shallow trees underfit; deep trees overfit.


In [ ]:
depths = list(range(1, 16))
train_accs = []
test_accs  = []

for d in depths:
    m = DecisionTreeClassifier(max_depth=d, criterion='gini', random_state=42)
    m.fit(X_train, y_train)
    train_accs.append(accuracy_score(y_train, m.predict(X_train)))
    test_accs.append(accuracy_score(y_test, m.predict(X_test)))

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(depths, train_accs, 'o-', color='#2196F3', label='Training Accuracy')
ax.plot(depths, test_accs,  's--', color='#FF5722', label='Testing Accuracy')
ax.axvline(x=5, color='green', linestyle=':', linewidth=1.5, label='Chosen depth = 5')
ax.set_xlabel('Tree Depth', fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Training vs. Testing Accuracy by Tree Depth', fontsize=13)
ax.legend(fontsize=11)
ax.grid(alpha=0.4)
ax.set_xticks(depths)
plt.tight_layout()
plt.savefig('../figures/depth_vs_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Best test accuracy: {max(test_accs):.4f} at depth {depths[test_accs.index(max(test_accs))]}")
print(f"Test accuracy at depth 5: {test_accs[4]:.4f}")


## 11. Delay Rate by Country

Before wrapping up, we visualize the proportion of delayed records per country in the full cleaned dataset.  
This contextualizes the model by showing real-world patterns across the 6 countries.


In [ ]:
country_delay = df.groupby('iso_country')['Is_Delayed'].agg(['mean', 'count']).reset_index()
country_delay.columns = ['Country', 'Delay Rate', 'Records']
country_delay['Delay Rate %'] = (country_delay['Delay Rate'] * 100).round(1)
country_delay = country_delay.sort_values('Delay Rate', ascending=False)

country_names = {'FR': 'France', 'DE': 'Germany', 'IT': 'Italy',
                 'ES': 'Spain', 'NL': 'Netherlands', 'GB': 'United Kingdom'}
country_delay['Country Name'] = country_delay['Country'].map(country_names)

fig, ax = plt.subplots(figsize=(9, 5))
colors_map = ['#e74c3c' if r > 0.3 else '#3498db' for r in country_delay['Delay Rate']]
bars = ax.bar(country_delay['Country Name'], country_delay['Delay Rate %'],
              color=colors_map, edgecolor='white', width=0.6)

for bar, val in zip(bars, country_delay['Delay Rate %']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val}%', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_ylabel('Delay Rate (%)', fontsize=12)
ax.set_title('Proportion of Delayed Records by Country', fontsize=13)
ax.set_ylim(0, country_delay['Delay Rate %'].max() + 8)
ax.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.savefig('../figures/delay_rate_by_country.png', dpi=150, bbox_inches='tight')
plt.show()

print(country_delay[['Country Name', 'Delay Rate %', 'Records']].to_string(index=False))


## 12. Discussion

### 12.1 Model Performance

The Decision Tree classifier with `max_depth=5` achieved strong results on a dataset of over 40,000 airport-day records.  
The high accuracy, precision, and recall indicate that the model can effectively distinguish delayed from on-time airport days using the engineered features.

The most important features — average departure schedule delay and average arrival schedule delay — confirm that the model is learning the right signals.  
These delay values represent structural, chronic delay patterns at airports and are strong predictors of whether a given airport-day will cross the 15-minute delay threshold.

### 12.2 Bias-Variance Trade-Off

The depth analysis showed that test accuracy stabilizes around depth 4–6.  
Beyond depth 6, the training accuracy continues to increase while test accuracy plateaus or slightly drops — a clear sign of **overfitting**.  
Choosing `max_depth=5` sits at a good balance point between the two.

### 12.3 Country-Level Insights

Some countries consistently show higher delay rates than others.  
This may reflect differences in air traffic density, infrastructure investment, weather exposure, or airline scheduling practices.  
These country-level differences are captured by the `iso_country` one-hot encoded features in the model.

### 12.4 Limitations

| Limitation | Explanation |
|---|---|
| **Aggregated data** | The dataset uses airport-day records, not individual flights. Predictions are about an airport's punctuality on a given day, not a specific passenger's flight. |
| **No real-time weather** | While the dataset spans two years, it does not include actual weather measurements. Weather is one of the main causes of flight delays. |
| **EUROCONTROL coverage** | The punctuality data covers only European airports in the EUROCONTROL network. Results may not generalize to airports outside this area. |
| **Single algorithm** | Only one model type (Decision Tree) was used, as required by the course. More complex models (Random Forest, Gradient Boosting) might perform better. |
| **Delay threshold** | The 15-minute threshold for `Is_Delayed` is a standard convention but is somewhat arbitrary. A different threshold would change the class balance and results. |


## 13. Conclusion

This project built a complete data science pipeline for flight delay prediction across six European countries.

**Key accomplishments:**
1. Merged EUROCONTROL punctuality data with OurAirports metadata to create a rich analytical dataset with over 40,000 rows and 21 features
2. Performed thorough data cleaning including outlier removal via IQR, missing value handling, and duplicate removal
3. Engineered meaningful features including season, weekend flag, and airport size classification
4. Trained a Decision Tree classifier that predicts airport-day delay status with strong accuracy
5. Identified average schedule delay as the most important predictor of delay status
6. Visualized results across countries, tree depths, and feature importances to explain the model's behavior

**Key takeaways:**
- Airport-level delay patterns are highly predictable from historical punctuality metrics
- Season and country both play a role, though they are less dominant than the raw delay measurements themselves
- A simple, interpretable model like a Decision Tree is sufficient for this level of aggregated airport data

Future work could extend this pipeline with real weather data, airline-level breakdowns, or individual flight records to achieve more granular and actionable predictions.


## 14. Final Summary Table

In [ ]:
summary = {
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1 Score'],
    'Value': [f'{accuracy:.4f}', f'{precision:.4f}', f'{recall:.4f}', f'{f1:.4f}']
}

summary_df = pd.DataFrame(summary)
print("\n=== Final Model Performance Summary ===")
print(summary_df.to_string(index=False))
print(f"\nModel: Decision Tree (max_depth=5, criterion=gini)")
print(f"Train size: {len(X_train):,} | Test size: {len(X_test):,}")
print(f"Features used: {X.shape[1]}")
print(f"Countries: FR, DE, IT, ES, NL, GB")
print(f"Target: Is_Delayed (threshold = 15 minutes)")
